# Fully connected networks, optimizers, and dropout

University of Michigan EECS 498/598, Winter 2022. Compact companion to the completed course assignment.

**Static validation only; not executed.** No ordinary-notebook outputs or scores are copied here. Run cells in order only when execution is intended; training variants retain the ordinary notebook parameters. Existing course dependencies and data are required; no installation or repository cloning is performed. Results are isolated under `core_results/fully_connected_networks`.

## Setup

In [ ]:
from pathlib import Path
import os
import sys
import importlib

candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/drive/MyDrive/umich-dlcv-wi2022')]
PROJECT_ROOT = next((p for p in candidates if (p / 'assignments' / 'a3' / 'fully_connected_networks_core.ipynb').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open the canonical project or assignment directory; mount Drive first in Colab.')
ASSIGNMENT_DIR = PROJECT_ROOT / 'assignments' / 'a3'
if 'eecs598' in sys.modules and Path(sys.modules['eecs598'].__file__).resolve().parent != (ASSIGNMENT_DIR / 'eecs598').resolve():
    raise RuntimeError('Restart the kernel before switching assignments: eecs598 differs across assignments.')
os.chdir(ASSIGNMENT_DIR)
sys.path.insert(0, str(ASSIGNMENT_DIR))
sys.modules.setdefault('imp', importlib)
import torch
import matplotlib.pyplot as plt
import eecs598
from eecs598 import reset_seed

device = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_DIR = ASSIGNMENT_DIR / 'core_results' / 'fully_connected_networks'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
reset_seed(0)
print('Device:', device)


## Data

In [ ]:
DATA_DIR = next((p for p in (ASSIGNMENT_DIR, PROJECT_ROOT / 'assignments' / 'a1') if (p / 'cifar-10-batches-py').is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Prepare the course CIFAR-10 cache before running this notebook.')
try:
    os.chdir(DATA_DIR)
    data_dict = eecs598.data.preprocess_cifar10(cuda=device == 'cuda', dtype=torch.float64, flatten=True, bias_trick=False, show_examples=False)
finally:
    os.chdir(ASSIGNMENT_DIR)

from eecs598 import Solver
def save_solver_histories(stage, trained_solvers):
    histories = {str(label): {'loss': trained.loss_history, 'train_accuracy': trained.train_acc_history, 'validation_accuracy': trained.val_acc_history} for label, trained in trained_solvers.items()}
    torch.save(histories, RESULTS_DIR / f'{stage}_histories.pt')
    for label, trained in trained_solvers.items():
        plt.plot(trained.val_acc_history, label=str(label))
    plt.xlabel('Epoch')
    plt.ylabel('Validation accuracy')
    plt.legend()
    plt.savefig(RESULTS_DIR / f'{stage}_validation.png', bbox_inches='tight')
    plt.show()


## Train the two-layer network

In [ ]:
from fully_connected_networks_core import create_solver_instance
reset_seed(0)
solver = create_solver_instance(data_dict=data_dict, dtype=torch.float64, device=device)
solver.train()
solver.model.save(str(RESULTS_DIR / 'best_two_layer_net.pth'))
print('Validation accuracy:', solver.check_accuracy(data_dict['X_val'], data_dict['y_val']))
print('Test accuracy:', solver.check_accuracy(data_dict['X_test'], data_dict['y_test']))
save_solver_histories('stage_41', {'model': solver})


## Overfit 50 examples with three layers

In [ ]:
from fully_connected_networks_core import FullyConnectedNet, get_three_layer_network_params
reset_seed(0)
num_train = 50
small_data = {'X_train': data_dict['X_train'][:num_train], 'y_train': data_dict['y_train'][:num_train], 'X_val': data_dict['X_val'], 'y_val': data_dict['y_val']}
weight_scale, learning_rate = get_three_layer_network_params()
model = FullyConnectedNet([100, 100], weight_scale=weight_scale, dtype=torch.float32, device=device)
solver = Solver(model, small_data, print_every=10, num_epochs=20, batch_size=25, optim_config={'learning_rate': learning_rate}, device=device)
solver.train()
plt.plot(solver.loss_history, 'o')
plt.title('Training loss history')
plt.xlabel('Iteration')
plt.ylabel('Training loss')
plt.show()
save_solver_histories('stage_49', {'model': solver})


## Overfit 50 examples with five layers

In [ ]:
from fully_connected_networks_core import FullyConnectedNet, get_five_layer_network_params
reset_seed(0)
num_train = 50
small_data = {'X_train': data_dict['X_train'][:num_train], 'y_train': data_dict['y_train'][:num_train], 'X_val': data_dict['X_val'], 'y_val': data_dict['y_val']}
weight_scale, learning_rate = get_five_layer_network_params()
model = FullyConnectedNet([100, 100, 100, 100], weight_scale=weight_scale, dtype=torch.float32, device=device)
solver = Solver(model, small_data, print_every=10, num_epochs=20, batch_size=25, optim_config={'learning_rate': learning_rate}, device=device)
solver.train(return_best_params=False)
plt.plot(solver.loss_history, 'o')
plt.title('Training loss history')
plt.xlabel('Iteration')
plt.ylabel('Training loss')
plt.show()
solver.model.save(str(RESULTS_DIR / 'best_overfit_five_layer_net.pth'))
save_solver_histories('stage_51', {'model': solver})


## SGD and momentum

In [ ]:
from fully_connected_networks_core import FullyConnectedNet, sgd, sgd_momentum
reset_seed(0)
num_train = 4000
small_data = {'X_train': data_dict['X_train'][:num_train], 'y_train': data_dict['y_train'][:num_train], 'X_val': data_dict['X_val'], 'y_val': data_dict['y_val']}
solvers = {}
for update_rule_name, update_rule_fn in [('sgd', sgd), ('sgd_momentum', sgd_momentum)]:
    print('running with ', update_rule_name)
    model = FullyConnectedNet([100, 100, 100, 100, 100], weight_scale=0.05, dtype=torch.float32, device=device)
    solver = Solver(model, small_data, num_epochs=5, batch_size=100, update_rule=update_rule_fn, optim_config={'learning_rate': 0.05}, print_every=1000, verbose=True, device=device)
    solvers[update_rule_name] = solver
    solver.train()
    print()
plt.subplot(3, 1, 1)
plt.title('Training loss')
plt.xlabel('Iteration')
for update_rule, solver in solvers.items():
    plt.plot(solver.loss_history, 'o', label='loss_%s' % update_rule)
plt.legend(loc='lower center', ncol=4)
plt.subplot(3, 1, 2)
plt.title('Training accuracy')
plt.xlabel('Epoch')
for update_rule, solver in solvers.items():
    plt.plot(solver.train_acc_history, '-o', label='train_acc_%s' % update_rule)
plt.legend(loc='lower center', ncol=4)
plt.subplot(3, 1, 3)
plt.title('Validation accuracy')
plt.xlabel('Epoch')
for update_rule, solver in solvers.items():
    plt.plot(solver.val_acc_history, '-o', label='val_acc_%s' % update_rule)
plt.legend(loc='lower center', ncol=4)
plt.gcf().set_size_inches(10, 20)
plt.show()
save_solver_histories('stage_58', solvers)


## Adam and RMSProp

In [ ]:
from fully_connected_networks_core import adam, rmsprop, FullyConnectedNet
for update_rule_name, update_rule_fn, learning_rate in [('adam', adam, 0.001), ('rmsprop', rmsprop, 0.0001)]:
    print('running with ', update_rule_name)
    model = FullyConnectedNet([100, 100, 100, 100, 100], weight_scale=0.05, device=device)
    solver = Solver(model, small_data, num_epochs=5, batch_size=100, update_rule=update_rule_fn, optim_config={'learning_rate': learning_rate}, print_every=1000, verbose=True, device=device)
    solvers[update_rule_name] = solver
    solver.train()
    print()
plt.subplot(3, 1, 1)
plt.title('Training loss')
plt.xlabel('Iteration')
for update_rule, solver in list(solvers.items()):
    plt.plot(solver.loss_history, 'o', label=update_rule)
plt.legend(loc='lower center', ncol=4)
plt.subplot(3, 1, 2)
plt.title('Training accuracy')
plt.xlabel('Epoch')
for update_rule, solver in list(solvers.items()):
    plt.plot(solver.train_acc_history, '-o', label=update_rule)
plt.legend(loc='lower center', ncol=4)
plt.subplot(3, 1, 3)
plt.title('Validation accuracy')
plt.xlabel('Epoch')
for update_rule, solver in list(solvers.items()):
    plt.plot(solver.val_acc_history, '-o', label=update_rule)
plt.legend(loc='lower center', ncol=4)
plt.gcf().set_size_inches(10, 20)
plt.show()
save_solver_histories('stage_64', solvers)


## Dropout and hidden-width comparison

In [ ]:
from fully_connected_networks_core import FullyConnectedNet
reset_seed(0)
num_train = 20000
small_data = {'X_train': data_dict['X_train'][:num_train], 'y_train': data_dict['y_train'][:num_train], 'X_val': data_dict['X_val'], 'y_val': data_dict['y_val']}
solvers = {}
dropout_choices = [0, 0, 0.5]
width_choices = [256, 512, 512]
for dropout, width in zip(dropout_choices, width_choices):
    model = FullyConnectedNet([width], dropout=dropout, dtype=torch.float32, device=device)
    print('Training a model with dropout=%.2f and width=%d' % (dropout, width))
    solver = Solver(model, small_data, num_epochs=100, batch_size=512, update_rule=adam, optim_config={'learning_rate': 0.005}, print_every=100000, print_acc_every=10, verbose=True, device=device)
    solver.train()
    solvers[dropout, width] = solver
    print()
save_solver_histories('stage_73', solvers)


## Evaluate dropout variants

In [ ]:
for setting, trained_solver in solvers.items():
    print(setting, 'validation:', trained_solver.check_accuracy(data_dict['X_val'], data_dict['y_val']), 'test:', trained_solver.check_accuracy(data_dict['X_test'], data_dict['y_test']))
    plt.plot(trained_solver.val_acc_history, label=str(setting))
plt.xlabel('Epoch')
plt.ylabel('Validation accuracy')
plt.legend()
plt.savefig(RESULTS_DIR / 'dropout_comparison.png', bbox_inches='tight')
plt.show()
